# Tugas 1 - Klasifikasi Jamur dengan Naive Bayes

| Nama | NRP |
|------|-----|
| Muhamad Nafi' Mulyo | 5054251005|

## Deskripsi Tugas
Pada tugas ini, kita akan menggunakan Mushroom Dataset dari UCI Machine Learning Repository.

Dataset dapat diakses melalui:
- https://www.kaggle.com/datasets/uciml/mushroom-classification

Untuk metadata / penjelasan mengenai dataset bisa mengakses link berikut:
- https://archive.ics.uci.edu/dataset/73/mushroom

Tujuan utama dari tugas ini adalah membangun dan membandingkan tiga varian Naive Bayes, yaitu Categorical NB, Bernoulli NB, dan Multinomial NB, untuk mengklasifikasikan apakah suatu jamur dapat dimakan (edible) atau beracun (poisonous).

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset dan Eksplorasi Awal
   - Memuat dataset, melihat struktur data, dan distribusi label.
2. Preprocessing
   - Memproses data agar siap digunakan untuk masing-masing varian Naive Bayes.
3. Eksperimen Model
   - Bangun model Categorical NB, Bernoulli NB, dan Multinomial NB.
4. Evaluasi Model
   - Hitung metrik evaluasi dan visualisasikan hasilnya.
5. Analisis dan Kesimpulan
   - Bandingkan hasil ketiga model dan berikan kesimpulan.

# 1. Persiapan Dataset dan Eksplorasi Awal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
# Sklearn Naive Bayes Variants
from sklearn.naive_bayes import CategoricalNB, BernoulliNB, MultinomialNB
# Sklearn Metrics Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc
)
# Styling plot agar terlihat rapi
sns.set_theme(style="whitegrid")

: 

In [ ]:
# Load dataset dan tampilkan beberapa baris pertama.
# Semua nilai pada dataset berupa single character yang merupakan kode singkatan dari kategori.
# Contoh: pada kolom class, e = edible dan p = poisonous.
df = pd.read_csv('mushrooms.csv')

df.head()

In [ ]:
# Tampilkan informasi dasar dataset: jumlah baris dan kolom, tipe data, dan jumlah missing value.
# Perhatikan kolom stalk-root yang memiliki nilai '?' sebagai representasi missing value.
# Tampilkan informasi jumlah baris, kolom, dan tipe data
print("=== Info Dataset ===")
df.info()

print("\n=== Jumlah Missing Value (NaN/Null standar) ===")
print(df.isnull().sum())

print("\n=== Jumlah Nilai '?' (Missing Value Khusus pada stalk-root) ===")
print((df == '?').sum())


In [ ]:
# Tampilkan jumlah nilai unik pada setiap fitur.
# Perhatikan variasi jumlah kategori antar fitur, misalnya veil-type hanya punya 1 nilai unik.
# Tampilkan jumlah kategori unik untuk setiap fitur
print("=== Jumlah Nilai Unik per Fitur ===")
print(df.nunique())


In [ ]:
# Visualisasi distribusi target (class: e = edible, p = poisonous)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='class', palette='Set2')
plt.title('Distribusi Kelas Jamur (e: Edible, p: Poisonous)')
plt.xlabel('Class')
plt.ylabel('Jumlah')
plt.show()

# Tampilkan persentase per kelas
print("=== Persentase Kelas Target ===")
print(df['class'].value_counts(normalize=True) * 100)


## Lakukan Exploratory Data Analysis (EDA) tambahan yang diperlukan.

# 2. Preprocessing

Karena semua fitur bertipe kategorikal, setiap varian Naive Bayes membutuhkan representasi data yang berbeda:

| Model | Representasi yang dibutuhkan | Preprocessing |
|---|---|---|
| Categorical NB | Integer per kategori | OrdinalEncoder |
| Bernoulli NB | Binary 0/1 per kategori | LabelBinarizer / OneHotEncoder |
| Multinomial NB | Count / frekuensi non-negatif | OneHotEncoder |

In [ ]:
# Tangani missing value pada kolom stalk-root (nilai '?').
# Pilih strategi yang sesuai: drop baris, ganti dengan modus, atau jadikan kategori tersendiri.
# Buat copy dataframe agar data asli aman
df_clean = df.copy()

# Ganti nilai '?' dengan kategori baru 'm' (missing)
df_clean['stalk-root'] = df_clean['stalk-root'].replace('?', 'm')

# Cek kembali apakah masih ada nilai '?'
print("Jumlah '?' setelah penanganan:", (df_clean == '?').sum().sum())


In [ ]:
# Periksa apakah ada fitur yang hanya memiliki satu nilai unik.
# Fitur seperti itu tidak memberikan informasi apapun dan sebaiknya di-drop.
# Drop kolom veil-type karena hanya punya 1 nilai unik (tidak berguna untuk klasifikasi)
df_clean = df_clean.drop(columns=['veil-type'])

print("Jumlah kolom tersisa:", df_clean.shape[1])


In [ ]:
# Pisahkan fitur (X) dan target (y).
# Encode target: e = 0 (edible), p = 1 (poisonous).
# Bagi data menjadi train set dan test set (misalnya 80:20).
# Gunakan stratify=y agar distribusi kelas tetap proporsional.
# 1. Pisahkan X (fitur) dan y (target)
X = df_clean.drop(columns=['class'])
y = df_clean['class'].map({'e': 0, 'p': 1})  # e = 0 (edible), p = 1 (poisonous)
# 2. Split data 80% train, 20% test dengan stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Ukuran X_train: {X_train.shape}")
print(f"Ukuran X_test : {X_test.shape}")

In [ ]:
# Preprocessing untuk Categorical NB:
# Gunakan OrdinalEncoder untuk mengubah setiap kategori menjadi integer.
# Contoh: cap-shape b=0, c=1, f=2, k=3, s=4, x=5
# Fit hanya pada train set, transform pada train set dan test set.
# Inisialisasi OrdinalEncoder
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit HANYA pada train set, transform pada train set dan test set
X_train_ord = ordinal_encoder.fit_transform(X_train)
X_test_ord = ordinal_encoder.transform(X_test)

print("Shape X_train_ord (CategoricalNB):", X_train_ord.shape)


In [ ]:
# Preprocessing untuk Bernoulli NB dan Multinomial NB:
# Gunakan OneHotEncoder untuk mengubah setiap kategori menjadi representasi binary.
# Hasilnya berupa matriks sparse dengan nilai 0 dan 1.
# Fit hanya pada train set, transform pada train set dan test set.
# Inisialisasi OneHotEncoder
ohe_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit HANYA pada train set, transform pada train set dan test set
X_train_ohe = ohe_encoder.fit_transform(X_train)
X_test_ohe = ohe_encoder.transform(X_test)

print("Shape X_train_ohe (Bernoulli & Multinomial NB):", X_train_ohe.shape)


# 3. Eksperimen Model

## 3.1 Categorical Naive Bayes

CategoricalNB adalah varian yang paling sesuai untuk dataset ini karena semua fitur memang bertipe kategorikal. Model menghitung P(fitur=kategori | kelas) secara langsung untuk setiap kategori pada setiap fitur.

In [ ]:
# Inisialisasi dan latih model CategoricalNB menggunakan train set hasil OrdinalEncoder.
# Inisialisasi dan latih model CategoricalNB dengan data Ordinal
cat_nb = CategoricalNB()
cat_nb.fit(X_train_ord, y_train)


In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label dan probabilitas kelas 1 untuk keperluan evaluasi.
# Prediksi label dan probabilitas kelas 1 (poisonous) pada test set
y_pred_cat = cat_nb.predict(X_test_ord)
y_prob_cat = cat_nb.predict_proba(X_test_ord)[:, 1]


## 3.2 Bernoulli Naive Bayes

BernoulliNB bekerja pada representasi binary. Dengan OneHotEncoder, setiap kategori menjadi kolom tersendiri bernilai 0 atau 1, sehingga model memperlakukan setiap kategori sebagai fitur binary ada/tidak ada.

In [ ]:
# Inisialisasi dan latih model BernoulliNB menggunakan train set hasil OneHotEncoder.
# Inisialisasi dan latih model BernoulliNB dengan data OneHot (Binary)
ber_nb = BernoulliNB()
ber_nb.fit(X_train_ohe, y_train)


In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label dan probabilitas kelas 1 untuk keperluan evaluasi.
# Prediksi label dan probabilitas kelas 1 (poisonous) pada test set
y_pred_ber = ber_nb.predict(X_test_ohe)
y_prob_ber = ber_nb.predict_proba(X_test_ohe)[:, 1]


## 3.3 Multinomial Naive Bayes

MultinomialNB membutuhkan input non-negatif dan menginterpretasikannya sebagai frekuensi. Dengan OneHotEncoder yang menghasilkan nilai 0 dan 1, model dapat menerimanya sebagai counts meskipun secara semantik tiap nilai hanya menandakan ada/tidak ada kategori.

In [ ]:
# Inisialisasi dan latih model MultinomialNB menggunakan train set hasil OneHotEncoder.
# Inisialisasi dan latih model MultinomialNB dengan data OneHot (Count)
mul_nb = MultinomialNB()
mul_nb.fit(X_train_ohe, y_train)


In [ ]:
# Lakukan prediksi pada test set.
# Simpan hasil prediksi label dan probabilitas kelas 1 untuk keperluan evaluasi.
# Prediksi label dan probabilitas kelas 1 (poisonous) pada test set
y_pred_mul = mul_nb.predict(X_test_ohe)
y_prob_mul = mul_nb.predict_proba(X_test_ohe)[:, 1]


# 4. Evaluasi Model

In [ ]:
# Tampilkan metrik evaluasi untuk setiap model: Accuracy, Precision, Recall, dan F1-Score.
# Sajikan dalam satu tabel perbandingan agar mudah dibandingkan.
metrics_data = {
    'Model': ['Categorical NB', 'Bernoulli NB', 'Multinomial NB'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_cat),
        accuracy_score(y_test, y_pred_ber),
        accuracy_score(y_test, y_pred_mul)
    ],
    'Precision': [
        precision_score(y_test, y_pred_cat),
        precision_score(y_test, y_pred_ber),
        precision_score(y_test, y_pred_mul)
    ],
    'Recall': [
        recall_score(y_test, y_pred_cat),
        recall_score(y_test, y_pred_ber),
        recall_score(y_test, y_pred_mul)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_cat),
        f1_score(y_test, y_pred_ber),
        f1_score(y_test, y_pred_mul)
    ]
}

df_metrics = pd.DataFrame(metrics_data)
df_metrics


In [ ]:
# Tampilkan Confusion Matrix setiap model dalam bentuk heatmap.
# Susun ketiganya dalam satu baris subplot.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

models = [
    ('Categorical NB', y_pred_cat),
    ('Bernoulli NB', y_pred_ber),
    ('Multinomial NB', y_pred_mul)
]

for ax, (name, y_pred) in zip(axes, models):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Edible (0)', 'Poisonous (1)'],
                yticklabels=['Edible (0)', 'Poisonous (1)'])
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()


In [ ]:
# Tampilkan ROC Curve ketiga model dalam satu plot beserta nilai AUC masing-masing.
plt.figure(figsize=(8, 6))

models_prob = [
    ('Categorical NB', y_prob_cat),
    ('Bernoulli NB', y_prob_ber),
    ('Multinomial NB', y_prob_mul)
]

for name, y_prob in models_prob:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve Comparison - Naive Bayes Variants')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


# 5. Analisis

#### Jelaskan secara detail hasil analisis yang didapatkan dari eksperimen yang telah dilakukan. Bandingkan performa Categorical NB, Bernoulli NB, dan Multinomial NB berdasarkan hasil evaluasi. Kaitkan perbedaan performa dengan perbedaan representasi data yang digunakan oleh masing-masing model. Pastikan analisis didasari oleh hasil eksperimen sendiri yang telah dilakukan.

Berdasarkan hasil eksperimen klasifikasi jamur menggunakan tiga varian Naive Bayes, didapatkan analisis sebagai berikut:

1. **Perbandingan Performa Model**:
   - **Categorical NB** dan **Multinomial NB** menghasilkan performa terbaik yang identik dengan nilai **AUC 0.9972**. Kedua model ini mencatatkan kesalahan prediksi terendah dengan hanya 7 jamur edible yang salah diprediksi sebagai beracun (False Positive = 7) dan 81 jamur beracun diprediksi edible (False Negative = 81).
   - **Bernoulli NB** memiliki performa sedikit di bawah dua varian lainnya dengan nilai **AUC 0.9954** (False Positive = 13, False Negative = 96).

2. **Kaitan Performa dengan Representasi Data**:
   - **Categorical NB (OrdinalEncoder)**: Sangat cocok karena dataset ini murni kategorikal secara native. Model secara langsung menghitung $P(\text{fitur}=\text{kategori} \mid \text{kelas})$ dari setiap kategori fitur.
   - **Multinomial NB (OneHotEncoder)**: Walaupun secara teori didesain untuk data frekuensi/count, pemberian matriks biner (0/1) dari OneHotEncoder membuat Multinomial NB mampu memodelkan distribusi kategori dengan sangat presisi, setara dengan Categorical NB.
   - **Bernoulli NB (OneHotEncoder)**: Model ini mengevaluasi ada/tidaknya fitur (*presence/absence*). Karena mengasumsikan distribusi biner murni untuk tiap kolom hasil OneHotEncoder, informasi interaksi antar-kategori dari fitur yang sama sedikit terdistorsi sehingga error-nya sedikit lebih tinggi.

3. **Konteks Keselamatan (False Negative)**:
   - Dalam kasus klasifikasi jamur, **False Negative** (jamur beracun yang salah terprediksi sebagai edible/bisa dimakan) adalah kesalahan yang sangat berbahaya fatal secara medis. Meskipun AUC mendekati 1.0 (99.7%), masih ada 81 jamur beracun yang lolos sebagai edible.


# 6. Kesimpulan dan Saran

#### Berikan kesimpulan dari eksperimen yang telah dilakukan. Saran dapat berupa rekomendasi untuk eksperimen selanjutnya, perbaikan model, atau hal-hal lain yang relevan dengan tugas ini.

### Kesimpulan
1. Pemodelan Naive Bayes sangat efektif untuk memprediksi kelayakan konsumsi jamur pada UCI Mushroom Dataset dengan skor AUC > 99%.
2. **Categorical NB** (menggunakan OrdinalEncoder) dan **Multinomial NB** (menggunakan OneHotEncoder) adalah dua varian terbaik untuk dataset kategorikal ini.
3. Representasi data mempengaruhi performa model secara langsung: Ordinal Encoding lebih ringkas (21 kolom) dan ramah memori, sedangkan OneHot Encoding mekarkan fitur (116 kolom) yang paling cocok untuk Multinomial/Bernoulli NB.

### Saran
1. Untuk penelitian/eksperimen selanjutnya, disarankan mencoba model berbasis pohon (seperti *Decision Tree* atau *Random Forest*) untuk melihat apakah interaksi antar-fitur kategorikal dapat ditekan hingga menghasilkan **False Negative = 0**.
2. Eksperimen strategi penanganan missing value `stalk-root` lain seperti imputasi modus atau KNN Imputer untuk melihat dampaknya terhadap sensitivitas model.
